In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import warnings

In [4]:
import os

os.listdir("data")

[]

In [5]:
print(os.getcwd())

D:\ChurnSense\notebooks


In [6]:
(os.listdir())

['.ipynb_checkpoints',
 '01_data_understanding.ipynb',
 '02_data_cleaning.ipynb',
 '03_model_improvement.ipynb',
 '04_model_tuning.ipynb',
 '05_Hyperparameter Tuning + Cross-Validation.ipynb',
 'data']

In [7]:
df = pd.read_csv("../data/raw/Telco_customer_churn.csv")
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [8]:
import os
for root, dirs, files in os.walk("."):
    for file in files:
        print(os.path.join(root,file))

.\01_data_understanding.ipynb
.\02_data_cleaning.ipynb
.\03_model_improvement.ipynb
.\04_model_tuning.ipynb
.\05_Hyperparameter Tuning + Cross-Validation.ipynb
.\.ipynb_checkpoints\01_data_understanding-checkpoint.ipynb
.\.ipynb_checkpoints\01_data_understanding.ipynb-checkpoint.ipynb
.\.ipynb_checkpoints\02_data_cleaning-checkpoint.ipynb
.\.ipynb_checkpoints\03_model_improvement-checkpoint.ipynb
.\.ipynb_checkpoints\04_model_tuning-checkpoint.ipynb
.\.ipynb_checkpoints\05_Hyperparameter Tuning + Cross-Validation-checkpoint.ipynb


In [9]:
DATA_PATH = ("../data/raw/Telco_customer_churn.csv")

In [10]:
df = pd.read_csv(DATA_PATH)

In [11]:
print("Dataset Loaded Sucessfully")

Dataset Loaded Sucessfully


In [12]:
df["Churn Value"].value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

In [13]:
df["Churn Value"].value_counts(normalize = True)*100

Churn Value
0    73.463013
1    26.536987
Name: proportion, dtype: float64

In [14]:
leakage_cols = ["Churn Label", "Churn Score", "Churn Reason","CLTV", "CustomerID"]
df_clean = df.drop(columns=leakage_cols)
print("Shape after removing leakage:", df_clean.shape)

Shape after removing leakage: (7043, 28)


In [15]:
df_clean.columns

Index(['Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude',
       'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents',
       'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service',
       'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
       'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing',
       'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value'],
      dtype='object')

In [16]:
y = df_clean["Churn Value"]
X = df_clean.drop("Churn Value", axis=1)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (7043, 27)
Target shape: (7043,)


In [17]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numerical columns: 6
Categorical columns: 21


In [18]:
X["Total Charges"].dtype

dtype('O')

In [19]:
X["Total Charges"] = pd.to_numeric(X["Total Charges"], errors="coerce")

In [20]:
X.isnull().sum().sort_values(ascending=False)

Total Charges        11
Country               0
Count                 0
City                  0
Zip Code              0
Lat Long              0
Latitude              0
Longitude             0
Gender                0
Senior Citizen        0
State                 0
Partner               0
Dependents            0
Phone Service         0
Tenure Months         0
Internet Service      0
Online Security       0
Online Backup         0
Multiple Lines        0
Device Protection     0
Tech Support          0
Streaming Movies      0
Streaming TV          0
Contract              0
Paperless Billing     0
Payment Method        0
Monthly Charges       0
dtype: int64

In [21]:
X["Total Charges"] = pd.to_numeric(X["Total Charges"], errors="coerce")

In [22]:
X["Total Charges"].dtype

dtype('float64')

In [23]:
X.isnull().sum().sort_values(ascending=False)

Total Charges        11
Country               0
Count                 0
City                  0
Zip Code              0
Lat Long              0
Latitude              0
Longitude             0
Gender                0
Senior Citizen        0
State                 0
Partner               0
Dependents            0
Phone Service         0
Tenure Months         0
Internet Service      0
Online Security       0
Online Backup         0
Multiple Lines        0
Device Protection     0
Tech Support          0
Streaming Movies      0
Streaming TV          0
Contract              0
Paperless Billing     0
Payment Method        0
Monthly Charges       0
dtype: int64

In [24]:
X = X.dropna()
y = y[X.index]   # Align target with remaining rows

print("New shape:", X.shape)

New shape: (7032, 27)


In [25]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (7032, 27)
Target shape: (7032,)


In [26]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical columns:", num_cols)
print("\nCategorical columns:", cat_cols)

Numerical columns: Index(['Count', 'Zip Code', 'Latitude', 'Longitude', 'Tenure Months',
       'Monthly Charges', 'Total Charges'],
      dtype='object')

Categorical columns: Index(['Country', 'State', 'City', 'Lat Long', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Phone Service', 'Multiple Lines',
       'Internet Service', 'Online Security', 'Online Backup',
       'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies',
       'Contract', 'Paperless Billing', 'Payment Method'],
      dtype='object')


In [27]:
X = pd.get_dummies(X, drop_first=True)

print("Shape after encoding:", X.shape)

Shape after encoding: (7032, 2813)


NameError: name 'cols_to_drop' is not defined

In [ ]:
X = pd.get_dummies(X, drop_first=True)

print("Shape after encoding:", X.shape)

In [ ]:
cols_to_drop = [
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
]

X = X.drop(columns=cols_to_drop)

In [ ]:
leakage_cols = [
    "Churn Label",
    "Churn Score",
    "Churn Reason",
    "CLTV",
    "CustomerID"
]

df_clean = df.drop(columns=leakage_cols)

In [ ]:
y = df_clean["Churn Value"]
X = df_clean.drop("Churn Value", axis=1)

In [ ]:
X["Total Charges"] = pd.to_numeric(X["Total Charges"], errors="coerce")

X = X.dropna()
y = y[X.index]

In [ ]:
cols_to_drop = [
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
]

X = X.drop(columns=cols_to_drop)

In [ ]:
X = pd.get_dummies(X, drop_first=True)

print("Shape after encoding:", X.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
print(num_cols)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_prob))